In [ ]:
# Colab / fresh environment: install the module (builds the C++ core, takes a few minutes the first time)
try:
    import sensor_fusion
except ImportError:
    %pip install -q git+https://github.com/Rudip1/learn-sensor-fusion

# 5 · Registration: ICP, NDT, LiDAR odometry and its drift

**Recap** ([theory](../../1_theory/05_registration.md)). Matched points align in closed form: SVD of the
cross-covariance, reflection excluded (5.2). ICP alternates nearest-neighbour matching and that alignment
(Algorithm 5.1); point-to-plane minimizes the distance along the target normals (5.4) with a linearized step
(5.5). Chaining scan-to-scan registrations gives LiDAR odometry (5.6), whose error grows with distance.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import sensor_fusion as sf
from sensor_fusion import cloud, reg, sim
from sensor_fusion.plotting import SERIES, use_style

use_style()
rng = np.random.default_rng(5)
scene = sim.demo_scene()


def h2(x, y, a):
    """2-D homogeneous transform."""
    c, s = np.cos(a), np.sin(a)
    return np.array([[c, -s, x], [s, c, y], [0, 0, 1.0]])


def planar_scan(x, y, a, seed, noise=0.01):
    T = np.eye(4)
    T[:2, :2] = h2(0, 0, a)[:2, :2]
    T[:3, 3] = [x, y, 1.0]
    model = sim.LidarModel.planar(720, 30.0)
    model.range_noise = noise
    return sim.simulate_lidar(scene, T, model, seed)["points"][:, :2]


A = planar_scan(0, 0, 0, 1)
truth = h2(0.5, 0.1, 0.03)
B = planar_scan(0.5, 0.1, 0.03, 2)
print(f"two planar scans, {len(A)} and {len(B)} points; true motion (0.5 m, 0.1 m, {np.degrees(0.03):.2f} deg)")

### ✏️ Exercise 1 — the closed form
Implement `my_best_rigid(src, dst)` with eq. (5.2) for (n, d) arrays, returning a $(d+1)\times(d+1)$ homogeneous
matrix. Do not forget the reflection.

In [ ]:
def my_best_rigid(src, dst):
    # ✏️ your code here
    raise NotImplementedError


In [ ]:
src = rng.normal(size=(30, 3))
R0 = np.linalg.qr(rng.normal(size=(3, 3)))[0]
R0 *= np.linalg.det(R0)                          # a proper rotation
dst = src @ R0.T + [1.0, -2.0, 0.5]
T = my_best_rigid(src, dst)
assert np.allclose(T[:3, :3], R0) and np.allclose(T, reg.best_rigid(src, dst))
flat = src.copy(); flat[:, 2] = 0
assert np.isclose(np.linalg.det(my_best_rigid(flat, flat * [-1, 1, 1])[:3, :3]), 1.0)
print("✅ my_best_rigid: exact, and a proper rotation even for a mirrored plane")

### ✏️ Exercise 2 — point-to-point ICP
Write `my_icp(source, target, T0, iters, gate)` for 2-D scans with Algorithm 5.1, using `cloud.KdTree` for the
matching and your `my_best_rigid`. Return the final transform.

In [ ]:
def my_icp(source, target, T0, iters=50, gate=2.0):
    # ✏️ your code here
    raise NotImplementedError


In [ ]:
T0 = truth @ h2(0.5, -0.5, np.radians(8))
mine = my_icp(B, A, T0)
ref = reg.icp(B, A, T0, "point_to_point", 50, 2.0)["T"]
E = np.linalg.inv(truth) @ mine
print(f"my ICP: {100 * np.linalg.norm(E[:2, 2]):.1f} cm, {np.degrees(np.arctan2(E[1, 0], E[0, 0])):.2f} deg from the truth")
assert np.allclose(mine, ref, atol=1e-4)
print("✅ my_icp matches reg.icp")

## 1. Point-to-point against point-to-plane

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
for i, metric in enumerate(("point_to_point", "point_to_plane")):
    r = reg.icp(B, A, T0, metric, 100, 2.0)
    E = np.linalg.inv(truth) @ r["T"]
    ax[0].semilogy(r["history"], color=SERIES[i], label=f"{metric}: {r['iterations']} it., {100 * np.linalg.norm(E[:2, 2]):.1f} cm off")
    moved = B @ r["T"][:2, :2].T + r["T"][:2, 2]
    if i == 1:
        ax[1].plot(A[:, 0], A[:, 1], ".", ms=2, color="0.6", label="target")
        ax[1].plot(moved[:, 0], moved[:, 1], ".", ms=2, color=SERIES[i], label="aligned source")
ax[0].set_xlabel("iteration"); ax[0].set_ylabel("RMS residual [m]"); ax[0].legend()
ax[1].set_aspect("equal"); ax[1].legend(); ax[1].set_title("point-to-plane result")
plt.show()

## 2. The basin of convergence

In [ ]:
def success_rate(metric, offset, yaw_deg, trials=30):
    ok = 0
    for _ in range(trials):
        d = rng.normal(size=2)
        d *= offset / np.linalg.norm(d)
        E = np.linalg.inv(truth) @ reg.icp(B, A, truth @ h2(*d, np.radians(yaw_deg) * rng.choice([-1, 1])), metric, 100, 2.0)["T"]
        ok += np.linalg.norm(E[:2, 2]) < 0.05 and abs(np.arctan2(E[1, 0], E[0, 0])) < np.radians(0.5)
    return ok / trials


for metric in ("point_to_point", "point_to_plane"):
    print(metric, [f"{success_rate(metric, o, y):.0%}" for o, y in ((1, 10), (2, 20), (3, 30))], "from 1 m/10°, 2 m/20°, 3 m/30°")

### ✏️ Exercise 3 — chaining
`reg.icp(scan_k, scan_{k-1})` returns $T_{k-1,k}$. Write `chain(steps)` that returns the poses $T_{0,k}$ with
eq. (5.6), and check it against `reg.lidar_odometry` on a simulated drive.

In [ ]:
def chain(steps):
    # ✏️ your code here
    raise NotImplementedError


In [ ]:
xs = np.linspace(-20, 20, 61)
ys = 1.5 * np.sin(xs / 8)
yaws = np.arctan(1.5 / 8 * np.cos(xs / 8))
scans = [planar_scan(x, y, a, i) for i, (x, y, a) in enumerate(zip(xs, ys, yaws))]
steps, guess = [], np.eye(3)
for k in range(1, len(scans)):
    guess = reg.icp(scans[k], scans[k - 1], guess, "point_to_plane", 50, 1.0)["T"]
    steps.append(guess)
mine = chain(steps)
odo = reg.lidar_odometry(scans, "point_to_plane", 50, 1.0)["poses"]
assert all(np.allclose(a, b, atol=1e-9) for a, b in zip(mine, odo))
print("✅ chain matches reg.lidar_odometry")

## 3. Drift

In [ ]:
T0inv = np.linalg.inv(h2(xs[0], ys[0], yaws[0]))
rel = np.array([(T0inv @ h2(x, y, a))[:2, 2] for x, y, a in zip(xs, ys, yaws)])
dist = np.r_[0, np.cumsum(np.linalg.norm(np.diff(rel, axis=0), axis=1))]
for i, metric in enumerate(("point_to_point", "point_to_plane")):
    p = np.array([T[:2, 2] for T in reg.lidar_odometry(scans, metric, 50, 1.0)["poses"]])
    plt.plot(dist, np.linalg.norm(p - rel, axis=1), color=SERIES[i], label=metric)
plt.xlabel("distance travelled [m]"); plt.ylabel("position error [m]"); plt.legend(); plt.show()

## 4. The recorded scans

In [ ]:
d = cloud.read_pcd(str(sf.data.path("parking_lot/run1_scans.pcd")))
real = [np.c_[d["x"][d["scan"] == k], d["y"][d["scan"] == k]] for k in range(int(d["scan"].max()) + 1)]
r = reg.lidar_odometry(real, "point_to_plane", 50, 0.5)
p = np.array([T[:2, 2] for T in r["poses"]])
print(f"{len(real)} scans: net displacement {np.linalg.norm(p[-1]):.1f} m, mean ICP residual {np.mean(r['rmse']):.3f} m")
plt.plot(p[:, 0], p[:, 1]); plt.gca().set_aspect("equal"); plt.title("odometry in the frame of the first scan"); plt.show()

## 🔨 Break it — no initial guess

Use only every 5th scan (3.4 m apart) or every 10th (6.8 m apart), with and without the previous motion as the
initial guess. ICP converges only inside its basin.

In [ ]:
for gap in (1, 5, 10):
    sub = scans[::gap]
    for cv in (True, False):
        p = np.array([T[:2, 2] for T in reg.lidar_odometry(sub, "point_to_plane", 50, 1.0, constant_velocity=cv)["poses"]])
        print(f"every {gap:2d}. scan, constant velocity {cv!s:5s}: end error {np.linalg.norm(p[-1] - rel[::gap][-1]):.2f} m")

## 🔨 Break it — chaining in the wrong order

In [ ]:
wrong = [np.eye(3)]
for T in steps:
    wrong.append(T @ wrong[-1])
print(f"end error, right order {np.linalg.norm(mine[-1][:2, 2] - rel[-1]):.2f} m, wrong order {np.linalg.norm(wrong[-1][:2, 2] - rel[-1]):.2f} m")

## What to remember

- Matched points align in closed form; exclude the reflection.
- ICP converges locally; give it a good initial guess (previous motion, IMU, wheels).
- Point-to-plane is faster and unbiased by the sampling pattern, but has a smaller basin.
- Odometry drifts; its error grows with distance and needs an absolute reference (chapter 6).